#Demo de Agente DQN para jugar a un juego de Atari

 Se utiliza un entorno Atari compatible para Gymnasium disponible en: https://ale.farama.org/environments/

In [ ]:
#@title Cargar Librerías


import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import random
from random import randint
import pandas as pd
import os

import math

from tqdm import tqdm
from typing import Optional
import gymnasium as gym

import ale_py
import io
import base64
from IPython.display import HTML

print("Librerías cargadas.")

## Clases sobre el Problema a resolver

In [ ]:
#@title Definir Entorno del Juego Atari

#
nombre_juego = "Breakout" #@param ["Breakout", "Galaxian",  "MsPacman", "Pong", "SpaceInvaders", "Tetris", "VideoChess", "VideoPinball"]
#@markdown nombre_juego: indica juego del entorno, para agregar otro ver https://ale.farama.org/environments/
tipo_obs_juego = "grayscale" #@param ["ram", "grayscale", "rgb"]
#@markdown tipo obs: indicar si se usa estado de la Memoria interna de la consola (RAM), Pantalla Color (RGB), o en Blanco & Negro  (grayscale)

# Inicializa el entorno de entrenamiento y evaluación
envJuegoName = "ALE/" + nombre_juego + "-v5"
gym.register_envs(ale_py)
envProblema = gym.make(envJuegoName,
                       obs_type=tipo_obs_juego,
                       frameskip=5,
                       repeat_action_probability=0.25,
                       full_action_space=False,
                       render_mode="rgb_array")

print("")
print("> Juego: ", nombre_juego, " [ ", envJuegoName, " ]")
print("\t-Tipo Observation: ", tipo_obs_juego)
print("\t-Observation space: ", envProblema.observation_space)
print("\t-Action space: ", envProblema.action_space)
print("\t-Tipo de Acciones posibles: ", envProblema.unwrapped.get_action_meanings())
print("")

# wrapper para grabar las jugadas
envDirVideo = "./gym-episode-video" # directorio para grabar video
grabaVideoEnv = False # variable que controla si se graba o no el video de un episodio
envProblema = gym.wrappers.RecordVideo(envProblema,
                                 video_folder=envDirVideo,
                                 episode_trigger=(lambda ep: grabaVideoEnv),
                                 video_length=10000,
                                 disable_logger=True)


# función auxiliarr para mostrar video de jugada
def showEnvVideo(env):
  global envDirVideo
  video_name = os.path.join(envDirVideo, f"{env.name_prefix}-episode-{env.episode_id}.mp4")
  #print(video_name)
  if os.path.exists(video_name):
    video = io.open(video_name, 'r+b').read()
    encoded = base64.b64encode(video)
    display(HTML(data='''
      <video width="360" height="auto" alt="test" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>'''
  .format(encoded.decode('ascii'))))
  else:
    print("-- No se encuentra video del episodio ", video_name)
  return

print("Entorno del Problema definido.")

# Clase para Agente que juega al azar
# lo único que considera son las posibles acciones a realizar
# para elegir una al azar
class RandomAgentClass():

  def __init__(self, observation_space, action_space):
    self._action_space = action_space

  def action(self, observation=None, info=None):
      # devuelve una acción al azar
      return self._action_space.sample()

randomAgent = RandomAgentClass(envProblema.observation_space,
                                                envProblema.action_space)
print("Agente Random definido para el entorno. ")

# funciones auxiliares para simular la ejecución del entorno

def mostrarObsInfoStep(env_obs, env_info, env_reward=None, action_step=None):
  print("\t", end=" ")
  if action_step is not None:
    print("action_step {}".format(action_step), end=" -> ")
  if env_reward is None:
      print("env obs {} & info {}".format(env_obs, env_info))
  else:
      print("env obs {} & info {} --> env reward {}".format(env_obs, env_info, env_reward))

# definir simulador para probar el entorno
def SimularEntorno(env, ag, titulo="", mostrarRender=True, mostrarResultado=True, mostarInfoStep=False) :
    # determina si se graba o no el enviroment
    global grabaVideoEnv
    bak_grabaVideoEnv = grabaVideoEnv
    grabaVideoEnv = mostrarRender
    if len(titulo)>0:
      print("\n** ", titulo, "**")
    # inicializa variables
    done = False
    # resetea el entorno
    env_obs, env_info  = env.reset()
    if mostarInfoStep:
      mostrarObsInfoStep(env_obs, env_info)
    while not done:
        # el agente determina la acción a realizar
        action_step = ag.action(env_obs, env_info)
        # se ejecuta la acción en el entorno
        env_obs, env_reward, env_terminated, env_truncated, env_info =  env.step(action_step)
        if mostarInfoStep:
          mostrarObsInfoStep(env_obs, env_info, env_reward, action_step)
        # determina si finaliza la simulación
        done = env_terminated or env_truncated
    if mostrarResultado:
      # muestra resultaodos finales
      print(" -Recompensa Final = " + str(round(env_reward, 3)))
    # cierra el enviroment
    env.close()
    if mostrarRender and grabaVideoEnv:
      # muestra video de jugada
      print(" -Video del episodio: ")
      showEnvVideo(env)
      print("")
    # vuelve al estado anterior de grabar video
    grabaVideoEnv = bak_grabaVideoEnv
    # devuelve el reward
    return env_reward

# función auxiliar para comparar
def compararRtdosAgentes(cantidad_probar, envProblema, ag1, ag2, descAg1="Agente 1", descAg2="Agente 2", mostrarRenderAg1=False, mostrarRenderAg2=False):
  prom1 = 0
  prom2 = 0
  for i in range(cantidad_probar):
    print("\n> Prueba ", i+1, ":")
    # Probar 1
    valor1 = SimularEntorno(envProblema, ag1, "Resultados de " + descAg1, mostrarRender=mostrarRenderAg1, mostrarResultado=True)
    prom1 = prom1 + valor1
    # Probar 2
    valor2 = SimularEntorno(envProblema, ag2, "Resultados de " + descAg2, mostrarRender=mostrarRenderAg2, mostrarResultado=True)
    prom2 = prom2 + valor2
    # Decide Ganador
    strMostrar = "\n--> " + descAg1 + " (%s) genera" % valor1
    if valor1 > valor2:
      strMostrar = strMostrar + " MEJOR "
    elif valor1 < valor2:
      strMostrar = strMostrar + " PEOR "
    else:
      strMostrar = strMostrar + " IGUAL "
    strMostrar = strMostrar + "resultado que " + descAg2 + " (%s)." % valor2
    print(strMostrar)
  # Decide Ganador General
  if cantidad_probar > 0:
    prom1 = prom1 / cantidad_probar
    prom2 = prom2 / cantidad_probar
    print("\n================================================================================================\n")
    strMostrar = " * En Promedio " + descAg1 + " (%s) genera" % prom1
    if prom1 > prom2:
      strMostrar = strMostrar + " MEJORES "
    elif prom1 < prom2:
      strMostrar = strMostrar + " PEORES "
    else:
      strMostrar = strMostrar + " IGUALES "
    strMostrar = strMostrar + "resultados que " + descAg2 + " (%s)." % prom2
    print(strMostrar)
    print("\n================================================================================================\n")

print("Simulador del entorno definido.")

# Probar el entorno definido con Política Aleatoria (opcional)
Probar_Entorno_Random = True #@param {type:"boolean"}
MostarDetalleSteps = False #@param {type:"boolean"}

if Probar_Entorno_Random:
  SimularEntorno(envProblema, randomAgent, "Probando el entorno del problema al Azar",
                 mostrarRender=True, mostarInfoStep=MostarDetalleSteps)
print("")



## Deep-Q-Network (DQN)

In [ ]:
#@title Define clase Agente DQN

import tensorflow as tf
from collections import deque

#@markdown ### Parámetros generales:
agent_config = "Personalizada" #@param ["DeepMind", "Personalizada"]

#@markdown ### Parámetros de las capas ConvNet:
dqn_convNet_usar = True #@param {type:"boolean"}
dqn_convNet_tamaño_kernel_N =  3 #@param {type:"integer"}
dqn_convNet_tamaño_pooling_M = 3 #@param {type:"integer"}
dqn_convNet_cantidad_capas_ocultas =  3#@param {type:"integer"}

#@markdown ### Parámetros de las capas Lineales:
dqn_lineal_cant_neuronas_capas_ocultas = '30' #@param {type:"string"}
#@markdown (Nota: se puede indicar Cantidad de neuronas, D para DropOut, BN para BatchNormalization)
dqn_lineal_tipo_funcion = 'relu' #@param ['exponential', 'linear', 'relu', 'sigmoid', 'tanh' ]
dqn_lineal_porc_capa_DropOut = 0.3 #@param {type:"number"}

#@markdown ### Parámetros del Optimizador:
opt_tipo = "Adam" #@param ["Gradiente Decreciente", "Adam", "Adadelta", "Adagrad", "Adamax", "RMSprop", "Momentum", "NAG", "Nadam"]
opt_learning_rate = 0.01 #@param {type: "number"}


# diccionario auxiliar para pasar configuracion QNetwork
config_q_network = {}

# tamaño de los kernels y pooling (para simplificar son todas iguales)
if agent_config.lower()=="deepmind":
    # usa red definido en paper original de DeepMind
    config_q_network["cnn_deepmind_config"] = True
else:
    # usa red con configuración definida por parámetros
    config_q_network["cnn_deepmind_config"] = False
    config_q_network["cnn_use"] = dqn_convNet_usar
    if dqn_convNet_tamaño_kernel_N<1:
      dqn_convNet_tamaño_kernel_N = 1
    config_q_network["cnn_kernel_shape"] = (dqn_convNet_tamaño_kernel_N)
    if dqn_convNet_tamaño_pooling_M<0:
      dqn_convNet_tamaño_pooling_M=0
    config_q_network["cnn_pooling_shape"] = (dqn_convNet_tamaño_pooling_M)

    # indica la configuración para la parte Encoder
    #   (cada elemento de las listas son la configuración de las capas Conv)
    if dqn_convNet_cantidad_capas_ocultas<1:
      dqn_convNet_cantidad_capas_ocultas = 1
    cnn_filters = []
    for i in range(dqn_convNet_cantidad_capas_ocultas, 0, -1):
      cnn_filters.append( 2**(i+2) )
    config_q_network["cnn_filters"] = cnn_filters

    # chequea configuración de drop out
    if dqn_lineal_porc_capa_DropOut <= 0:
      dqn_lineal_porc_capa_DropOut = 0.10
    elif dqn_lineal_porc_capa_DropOut > 0.9:
        dqn_lineal_porc_capa_DropOut = 0.9
    config_q_network["lineal_porc_capa_DropOut"] = dqn_lineal_porc_capa_DropOut

    # cantidad de neuronas ocultas
    hidden_layers = []
    for val in dqn_lineal_cant_neuronas_capas_ocultas.split(','):
      val = val.strip()
      if val == "D":
        hidden_layers.append( "DropOut" )
      elif val == "BN":
        hidden_layers.append( "BatchNormalization" )
      elif val.isnumeric():
        hidden_layers.append( val )
      else:
        print("Capa ", val, "descartada!")
    config_q_network["hidden_layers"] = hidden_layers
    if (dqn_lineal_tipo_funcion is None) or  (len(dqn_lineal_tipo_funcion)==0):
      config_q_network["hidden_layers_func"]= "relu"
    else:
      config_q_network["hidden_layers_func"]= dqn_lineal_tipo_funcion

    # algoritmo de optimización
    if opt_tipo == "Gradiente Decreciente":
      config_q_network["optAlg"] = tf.keras.optimizers.SGD(learning_rate=opt_learning_rate)
    elif opt_tipo == "Adam":
      config_q_network["optAlg"] = tf.keras.optimizers.Adam(learning_rate=opt_learning_rate)
    elif opt_tipo == "Adadelta":
      config_q_network["optAlg"] = tf.keras.optimizers.Adadelta(learning_rate=opt_learning_rate)
    elif opt_tipo == "Adagrad":
      config_q_network["optAlg"] = tf.keras.optimizers.Adagrad(learning_rate=opt_learning_rate)
    elif opt_tipo == "Adamax":
      config_q_network["optAlg"] = tf.keras.optimizers.Adamax(learning_rate=opt_learning_rate)
    elif opt_tipo == "Nadam":
      config_q_network["optAlg"] = tf.keras.optimizers.Nadam(learning_rate=opt_learning_rate)
    elif opt_tipo == "RMSprop":
      config_q_network["optAlg"] = tf.keras.optimizers.RMSprop(learning_rate=opt_learning_rate)
    elif opt_tipo == "Momentum":
      config_q_network["optAlg"] = tf.keras.optimizers.SGD(learning_rate=opt_learning_rate, momentum=0.9, nesterov=False)
    elif opt_tipo == "NAG":
      config_q_network["optAlg"] = tf.keras.optimizers.SGD(learning_rate=opt_learning_rate, momentum=0.9, nesterov=True)
    else:
      config_q_network["optAlg"] = tf.keras.optimizers.Adam()

# Define clase auxiliar para memoria de Agente DQN en entrenamiento
class DQNReplayBuffer(object):

  def __init__(self, maxlen):
    self.buffer = deque(maxlen=maxlen)

  def reset(self):
    # resetea el buffer
    self.buffer.clear()

  def add(self, state, action, reward, next_state, done):
    # se fija si tiene que convertirlos a vectores
    if np.isscalar(state):
      state = [state]
    if np.isscalar(action):
      action = [action]
    if np.isscalar(next_state):
      next_state = [next_state]
    # agrega al buffer
    self.buffer.append((state, action, reward, next_state, done))

  def count(self):
    # devuelve la cantidad disponible
    return len(self.buffer)

  def sample(self, num_samples):
    # se fija si tiene suficientes ejemplos para devolver
    num_samples = max(num_samples, self.count())
    # devuelve una selección aleatoria de ejemplos del buffer
    states, actions, rewards, next_states, dones = [], [], [], [], []
    idx = np.random.choice(len(self.buffer), num_samples)
    for i in idx:
      elem = self.buffer[i]
      state, action, reward, next_state, done = elem
      states.append(state)
      actions.append(action)
      rewards.append(reward)
      next_states.append(next_state)
      dones.append(done)
    return states, actions, rewards, next_states, dones


# Define clase de Agente DQN
class DQNAgentClass():

  def __init__(self, observation_space, action_space, config_q_network):
      # define parámetros de capas de entrada/salida
      self._action_space = action_space
      self._num_actions = self._action_space.n
      self._observation_space = observation_space
      if len(self._observation_space.shape)==0:
        self._inputs_shape = (1,)
      else:
        self._inputs_shape = self._observation_space.shape
      # resetea hiperparámetros
      self.resetHyperparams()
      # Define configuración de la red
      self._config_q_network = config_q_network
      # construye la red
      self._create_net_models()

  # resetea hiperparámetros
  def resetHyperparams(self, gamma=0.99, reward_update_rate=100):
      # Hyperparameters
      # grado de descuento de recompensa
      self.gamma = max(gamma, 0.6)
      # valor de epsilon inicial para realizar exploración aleatoria
      self.epsilon = 1.0
      # cantidad de pasos de entrenamiento ejecutados
      self._train_steps = 0
      # cantida de pasos que deben pasar para actualizar los pesos de Targetnet
      self._update_rate = reward_update_rate

  # degrada epsilon hasta valor mínimo
  def degradeEpsilon(self, decay=0.995, min=0.01):
    if self.epsilon > epsilon_min:
        self.epsilon *= decay

  # crea red Q y red Reward (la segunda es un clon de la primera)
  def _create_net_models(self):
      if self._config_q_network["cnn_deepmind_config"]:
          # crea modelos usando arquitectura de paper de Deepmind
          self.q_model = self._create_q_model_deepmind("DQN",
                                      self._inputs_shape,
                                      self._num_actions)
      else:
          # crea modelos usando arquitectura a partir de configuración de usuario
          self.q_model = self._create_q_model_custom("DQN",
                                      self._inputs_shape,
                                      self._num_actions)
      # copia Qnet para reward Targetnet
      self.target_model = tf.keras.models.clone_model(self.q_model)
      self.target_model._name = "Reward-Model"
      self._update_target_model_weights()
      # determina optimizer para modelos
      if "optAlg" in config_q_network:
        _optimizer = config_q_network["optAlg"]
      else:
        _optimizer = tf.keras.optimizers.Adam(learning_rate=0.00025, clipnorm=1.0)
      self.q_model.compile(loss='mse', optimizer=_optimizer)
      # muestra uno de los dos modelos creados (tiene la misma estructura)
      self.q_model.summary()


  # Actualiza los pesos de Targetnet en base a Qnet
  def _update_target_model_weights(self):
      self.target_model.set_weights(self.q_model.get_weights())

  # Determina forma para preparar datos no matriciales
  def _defineInputReshape(self, inputShape):
      matrixShape = None
      for val in [2, 3, 5, 7, 11]:
        if (inputShape[0]%val)==0:
          matrixShape = [val,(inputShape[0]//val)]
          break
      if matrixShape is None:
        matrixShape = [1, inputShape[0]]
      return matrixShape

  # Crea Red basada en configuración de paper de DeepMind
  def _create_q_model_deepmind(self, modelName, inputShape, num_actions):
      # capa de entrada
      inputLay = tf.keras.layers.Input(shape=inputShape, name="input")
      eachLay = inputLay
      # agrega capas conv según corresponda
      if len(inputShape)==1:
        # ajusta la forma de entrada para poder usar capas conv1D
        eachLay = tf.keras.layers.Reshape(self._defineInputReshape(inputShape), name="reshape_input" )(eachLay)
        # agrega solo 1 capa convolution
        eachLay = tf.keras.layers.Conv1D(32, 2, strides=1, activation="relu", name="conv_1")(eachLay)
      elif len(inputShape)==2:
        # agrega 3 capas convolucionales 1D
        eachLay = tf.keras.layers.Conv1D(32, 2, strides=4, activation="relu", name="conv_1")(eachLay)
        eachLay = tf.keras.layers.Conv1D(64, 4, strides=2, activation="relu", name="conv_2")(eachLay)
        eachLay = tf.keras.layers.Conv1D(64, 3, strides=1, activation="relu", name="conv_3")(eachLay)
      else:
        # agrega 3 capas convolucionales 2D
        eachLay = tf.keras.layers.Conv2D(32, (2,2), strides=4, activation="relu", name="conv_1")(eachLay)
        eachLay = tf.keras.layers.Conv2D(64, (4,4), strides=2, activation="relu", name="conv_2")(eachLay)
        eachLay = tf.keras.layers.Conv2D(64, (3,3), strides=1, activation="relu", name="conv_3")(eachLay)
      # capas flatten y lineal
      eachLay = tf.keras.layers.Flatten(name="flat")(eachLay)
      eachLay = tf.keras.layers.Dense(512, activation="relu", name="lineal")(eachLay)
      # capa de salida
      outputLay = tf.keras.layers.Dense(num_actions, activation=None, name="output")(eachLay)
      # devuelve el modelo creado
      modelo = tf.keras.Model(name=modelName+"_DeepMind", inputs=inputLay, outputs=outputLay)
      return modelo


  # Crea Red personalizada definida por configuración
  def _create_q_model_custom(self, modelName, inputShape, num_actions):
      # capa de entrada
      inputLay = tf.keras.layers.Input(shape=inputShape, name="input")
      eachLay = inputLay
      if self._config_q_network["cnn_use"]:
        if len(inputShape)==1:
            # ajusta la forma de entrada para poder usar capas conv1D
            eachLay = tf.keras.layers.Reshape(self._defineInputReshape(inputShape), name="reshape_input" )(eachLay)
        # agrega capas convolucionales
        auxName = 'conv_'
        for i in range(len(self._config_q_network["cnn_filters"])):
            # define el nombre de la capa oculta
            auxlayerName = 'conv_'+str(i+1)
            # agrega las capas ocultas de tipo Conv2D
            if len(inputShape)>2:
              eachLay =  tf.keras.layers.Conv2D(self._config_q_network["cnn_filters"][i], self._config_q_network["cnn_kernel_shape"], activation='relu', padding='same', name='c_'+auxlayerName)(eachLay)
            else:
              eachLay =  tf.keras.layers.Conv1D(self._config_q_network["cnn_filters"][i], self._config_q_network["cnn_kernel_shape"], activation='relu', padding='same', name='c_'+auxlayerName)(eachLay)
            # determina nombre y shape de la capa conv2D
            last_conv_layer_name = 'c_'+auxlayerName
            if self._config_q_network["cnn_pooling_shape"] > 0:
              # sino no agrega capa MaxPooling
              if len(inputShape)>2:
                eachLay =  tf.keras.layers.MaxPooling2D(self._config_q_network["cnn_pooling_shape"], padding='same', name='p_'+auxlayerName)(eachLay)
              else:
                eachLay =  tf.keras.layers.MaxPooling1D(self._config_q_network["cnn_pooling_shape"], padding='same', name='p_'+auxlayerName)(eachLay)
        #  agrega capa Flatten
        eachLay = tf.keras.layers.Flatten(name='flat')(eachLay)
      elif len(inputShape)>1:
        #  agrega capa Flatten por entrada no lineal
        eachLay = tf.keras.layers.Flatten(name='flat')(eachLay)
      # agrega capas lineales
      auxName = 'lineal_'
      auxId = 1
      for val_hid in self._config_q_network["hidden_layers"]:
        if val_hid == "DropOut":
          auxlayerName = "d_"+str(auxId)
          auxId = auxId + 1
          eachLay =  tf.keras.layers.Dropout(self._config_q_network["lineal_porc_capa_DropOut"], name=auxlayerName)(eachLay)
        elif val_hid == "BatchNormalization":
          auxlayerName = "bn_"+str(auxId)
          auxId = auxId + 1
          eachLay =  tf.keras.layers.BatchNormalization(name=auxlayerName)(eachLay)
        elif val_hid.isnumeric():
          # agrega la capa oculta
          auxlayerName = auxName+str(auxId)
          auxId = auxId + 1
          eachLay =  tf.keras.layers.Dense(int(val_hid), activation=config_q_network["hidden_layers_func"], name=auxlayerName)(eachLay) # capas ocultas
      # capa de salida
      outputLay = tf.keras.layers.Dense(num_actions, activation=None, name="output")(eachLay)
      # devuelve el modelo creado
      modelo = tf.keras.Model(name=modelName+"_custom", inputs=inputLay, outputs=outputLay)
      return modelo

  # Ejecuta el una iteración de entrenamiento el modelo
  # usando los datos seleccionados aleatoriamente
  def train_step(self, sample_states, sample_actions, sample_rewards, sample_next_states, sample_done):
      # desempaqueta las listas
      sample_states = np.array(sample_states)
      sample_actions = np.array(sample_actions)
      sample_rewards = np.array(sample_rewards, dtype=np.float32)
      sample_next_states = np.array(sample_next_states)
      sample_done = np.array(sample_done)
      # determina prediccion de reward usando Targetnet
      model_rewards = self.target_model.predict(sample_next_states, verbose=0)
      # determina predicción de valores Q actualies usando Qnet
      model_qvalues = self.q_model.predict(sample_states, verbose=0)
      # realiza cálculo de rewards
      train_qvalues = []
      for i in range(len(model_rewards)):
          if sample_done[i]:
              target = sample_rewards[i]
          else:
              pred_reward = np.amax(model_rewards[i])
              target = sample_rewards[i] + self.gamma * pred_reward
          # actualiza el valor Q para la acción correspondiente
          model_qvalues[i][sample_actions[i]] = target
          # Use vectors in the objective computation
          train_qvalues.append( model_qvalues[i] )
      # reentrena el modelo
      self.q_model.fit(sample_states, np.array(train_qvalues), epochs=1, verbose=0)
      # libera memoria
      del sample_next_states
      del sample_actions
      del sample_rewards
      del sample_done
      del sample_states
      del train_qvalues
      # determina si tiene que actualizar modelo target
      self._train_steps += 1
      if (self._train_steps % self._update_rate) == 0:
          self._update_target_model_weights()


  # graba checkpoint temporal de los pesos del modelo
  def save_checkpoint(self, folder='checkpoints', filename='cp.weights.h5'):
    # nota se hace una pequeña modificación para que funcione corretamente
      if ".weights.h5" not in filename:
        filename = filename + ".weights.h5"
      if not os.path.exists(folder):
          #print("  No existe el directorio de checkpoints! Se crea {}".format(folder))
          os.mkdir(folder)
      self.q_model.save_weights(os.path.join(folder, "Q_"+filename))
      self.target_model.save_weights(os.path.join(folder, "T_"+filename))
      #print("  Checkpoint grabado en '{}'".format(filepath))

  # recupera checkpoint temporal de los pesos del modelo
  def load_checkpoint(self, folder='checkpoints', filename='cp.weights.h5'):
    # nota se hace una pequeña modificación para que funcione corretamente
      if ".weights.h5" not in filename:
        filename = filename + ".weights.h5"
      fnQmodel = os.path.join(folder, "Q_"+filename)
      fnTmodel = os.path.join(folder, "T_"+filename)
      if not os.path.exists(fnQmodel):
        print("  No se encuentra un checkpoint en '{}'".format(fnQmodel))
      elif not os.path.exists(fnTmodel):
        print("  No se encuentra un checkpoint en '{}'".format(fnTmodel))
      else:
        self.q_model.load_weights(fnQmodel)
        self.target_model.load_weights(fnTmodel)
        #print("  Checkpoint cargado de '{}'".format(filepath))

  # graba modelo completo
  def save_model(self, folder='models', filename='model.keras'):
    # nota nuevo método para guardar todo el modelo
      if not os.path.exists(folder):
          print("  No existe el directorio para guardar el modelo! Se crea {}".format(folder))
          os.mkdir(folder)
      self.q_model.save(os.path.join(folder, "Q_"+filename))
      self.target_model.save(os.path.join(folder, "T_"+filename))
      print("  Modelo grabado en '{}'".format(folder))

  # recupera modelo grabado
  def load_model(self, folder='models', filename='model.keras'):
    # nota nuevo método para cargar todo el modelo
      if not os.path.exists(folder):
        print("  No se encuentra el modelo guardado en '{}'".format(folder))
      else:
        fnQmodel = os.path.join(folder, "Q_"+filename)
        fnTmodel = os.path.join(folder, "T_"+filename)
        if not os.path.exists(fnQmodel):
          print("  No se encuentra un modelo en '{}'".format(fnQmodel))
        elif not os.path.exists(fnTmodel):
          print("  No se encuentra un modelo en '{}'".format(fnTmodel))
        else:
          self.q_model = tf.keras.models.load_model(fnQmodel)
          self.target_model = tf.keras.models.load_model(fnTmodel)
          print("  Modelo cargado de '{}'".format(folder))
          # muestra el modelo creado
          #self.model.summary()


  # Elije la acción durante proceso de entrenamiento
  # basdo en política "epsilon-greedy"
  def action_training(self, observation, info=None):
      # compara con el coeficiente epsilon como generar acción en training
      if random.uniform(0.0, 1.0) <= self.epsilon:
          # determina una acción al azar
          return self._action_space.sample()
      else:
          # predice la acción usando predicción de QNetwork
          return self.action(observation, info)

  # Elije siempre la mejor acción en base al estado actual
  # realizando la predicción con la red Q
  def action(self, observation, info=None):
      # devuelve valor determinado por Qnet
      if np.isscalar(observation):
        observation = [observation]
      act_values = self.q_model.predict(np.array( [ observation ] ), verbose=0)
      # utiliza el ID acción con mayor valor Q
      return np.argmax(act_values[0])

print("\nClase DQNAgentClass definida.")

# inicializa Agente DQN
dqnAg = DQNAgentClass(envProblema.observation_space,
                      envProblema.action_space,
                      config_q_network)

print("\nAgente DQN inicializado. ")


In [ ]:
#@title Entrenar al Agente DQN

from tqdm import tqdm
import time

#@markdown Parámetros del entrenamiento:
entrenar_DQN = True #@param {type:"boolean"}
#@markdown Hiper-parámetros:
gamma = 0.9 #@param {type:"number"}
epsilon_decay = 0.95 #@param {type:"number"}
epsilon_min = 0.3 #@param {type:"number"}
#@markdown Ciclos:
cant_episodios_entrenamiento_finalizar = 10 #@param {type:"integer"}
ajustar_reward_mode_cada = 10 #@param {type:"integer"}
#@markdown Datos para Entrenamiento:
replay_buffer_max_size = 1000 #@param {type:"integer"}
batch_size_entrenamiento = 64 #@param {type:"integer"}
pregenerar_datos_azar = True #@param {type:"boolean"}
#@markdown Evaluación:
calcular_recompensa_cada = 1 #@param {type:"integer"}
calcular_recompensa_si_mejora = True #@param {type:"boolean"}
cant_episodios_evaluacion = 1 #@param {type:"integer"}
mostrar_detalle_recompensa_promedio = True #@param {type:"boolean"}
minima_recompensa_promedio_finalizar = 0.99 #@param {type:"number"}

# controla valores
cant_episodios_entrenamiento_finalizar = max(1, cant_episodios_entrenamiento_finalizar)
ajustar_reward_mode_cada = max(5, ajustar_reward_mode_cada)
calcular_recompensa_cada = max(5, calcular_recompensa_cada)
cant_episodios_evaluacion = max(1, cant_episodios_evaluacion)
batch_size_entrenamiento = max(batch_size_entrenamiento, 5)

# Define Métricas para evaluación para Agente DQN
# Se usa el promedio de la recompensa (la más común)
def compute_avg_return(environment, agent, num_episodes=10, display=False):
  if num_episodes < 1:
      return 0.0
  total_return = 0.0
  reward = 0.0
  if display:
    print(" evaluando...", end="")
  for it in range(num_episodes):
    obs, info = environment.reset()
    done = False
    while not done:
      action_step = agent.action(obs)
      obs, reward, terminated, truncated, _  = environment.step(action_step)
      done = terminated or truncated
    if display:
      print(it+1, ":", reward, end=" ")
    total_return += reward
  avg_return = total_return / num_episodes
  print("")
  return round(avg_return,3)


# generación de datos usando agente azar
def generarDatosAzar(replayBuffer, env, ag, cant):
  for i in tqdm(range(1, cant+1)):
      done = False
      obs, info = env.reset()
      while not done:
          # ejecuta la acción del agente (versión de entrenamiento)
          action = ag.action(obs)
          # ejecuta la acción
          next_obs, reward, terminated, truncated, _ = env.step(action)
          # termino realmente el juego
          # o se forzo la terminación por máximos pasos
          done = terminated or truncated
          if not truncated:
            # graba en secuencia para entrenar
            replayBuffer.add(obs, action, reward, next_obs, terminated)
          # actualiza siguiente obs
          obs = next_obs


if entrenar_DQN:
  # deshabilita el grabado de video
  bak_grabaVideoEnv = grabaVideoEnv
  grabaVideoEnv = False
  # usa el ambiente para entrenar
  env = envProblema
  # crea replay buffer
  replayBuffer = DQNReplayBuffer(maxlen=max(replay_buffer_max_size, 1000))
  if pregenerar_datos_azar:
    print("\n ** Pre-generando datos aleatoriamente 1/4 del buffer **")
    generarDatosAzar(replayBuffer, env, randomAgent, replay_buffer_max_size//4)
  # resetea hiperparámetros del agente
  dqnAg.resetHyperparams(gamma=gamma,
                         reward_update_rate=calcular_recompensa_cada)
  # variables auxiliares
  buffer_alcanza_batch = False
  tempCheckpointDir = "./DQN_checkpoints/"
  best_avg_reward = None
  rewards_per_episode = []
  print("\n** Comienza Entrenamiento **")
  for ep in tqdm(range(1, cant_episodios_entrenamiento_finalizar+1)):
      reward = 0.0
      done = False
      obs, info = env.reset()
      while not done:
          # ejecuta la acción del agente (versión de entrenamiento)
          action = dqnAg.action_training(obs)
          # ejecuta la acción
          next_obs, reward, terminated, truncated, _ = env.step(action)
          # termino realmente el juego
          # o se forzo la terminación por máximos pasos
          done = terminated or truncated
          if not truncated:
            # graba en secuencia para entrenar
            replayBuffer.add(obs, action, reward, next_obs, terminated)
            if not buffer_alcanza_batch:
              # chequea si se alcanza cantidad mínima de ejemplos para entrenar
              if (replayBuffer.count() > batch_size_entrenamiento):
                buffer_alcanza_batch = True
          # actualiza nuevo estado
          obs = next_obs
          # entrena modelos usando estados anteriores
          if buffer_alcanza_batch:
              t1 = time.time()
              # obtiene un conjunto de ejemplos aleatorios para re-entrenar
              sample_states, sample_actions, sample_rewards, sample_next_states, sample_done = replayBuffer.sample(batch_size_entrenamiento)
              # manda a entrenar al modelo
              dqnAg.train_step(sample_states, sample_actions, sample_rewards, sample_next_states, sample_done)
              t2 = time.time()
              if (t2 - t1) > 30:
                 print("# warning procesando episodio {}/{}: replay y train_step tarda mucho!! {}"
                    .format(ep, cant_episodios_entrenamiento_finalizar, round(t2-t1,3)))
      # agrega para graficar
      rewards_per_episode.append(reward)
      # degrada episilon
      dqnAg.degradeEpsilon(decay=epsilon_decay, min=epsilon_min)
      # determina si tiene que realizar la evaluación forzada
      fuerzaEvalModelo = False
      if calcular_recompensa_si_mejora and (ep>1):
        # si el reward actual es mayor al anterior
        if (reward>last_reward):
          # y el reward actual es mayor al promedio del mejor (si está definido)
          if (best_avg_reward is None) or (reward>best_avg_reward):
            # fuerza evaluación
            fuerzaEvalModelo = True
      last_reward = reward
      # determina si tiene que realizar la evaluación
      if fuerzaEvalModelo or (ep == 1) or\
       (ep == cant_episodios_entrenamiento_finalizar) or\
        ((ep % calcular_recompensa_cada) == 0):
          # si se fuerza evaluación
          if fuerzaEvalModelo:
            printTextoFM = "--FE-"
          else:
            printTextoFM = "--"
          # calcula promedio de evaluación
          avg_reward = compute_avg_return(env, dqnAg, cant_episodios_evaluacion, display=mostrar_detalle_recompensa_promedio)
          print(printTextoFM, 'episodio {}/{}: Promedio Recompensa = {:.3f}'
                    .format(ep, cant_episodios_entrenamiento_finalizar, avg_reward))
          if (avg_reward >= minima_recompensa_promedio_finalizar):
              print('++ Finaliza en episodio {}/{} por buen valor de recompensa promedio: {:.3f}'
                  .format(ep, cant_episodios_entrenamiento_finalizar, avg_reward))
              break
          # chequea si el reward es mejor que best (si está definido)
          if (best_avg_reward is None) or (best_avg_reward<avg_reward):
            # guarda checkpoint actual
            if (best_avg_reward is not None):
              print("++ Guarda checkpoint mejor modelo: Best {:.3f} <= Actual {:.3f}"
                .format(best_avg_reward, avg_reward))
            dqnAg.save_checkpoint(folder=tempCheckpointDir, filename="best")
            best_avg_reward = avg_reward

  print("\n** Entrenamiento Finalizado *\n")
  # guarda último agente entrenado
  dqnAg.save_checkpoint(folder=tempCheckpointDir, filename="last")
  if ep>1:
    print("-- re-evaluando útimo modelo vs mejor modelo...")
    # vuelve a calcular reward
    if mostrar_detalle_recompensa_promedio:
      print("   -Último Modelo: ", end="")
    last_avg_reward = compute_avg_return(env, dqnAg, cant_episodios_evaluacion, display=mostrar_detalle_recompensa_promedio)
    # recupera checkpoint best
    dqnAg.load_checkpoint(folder=tempCheckpointDir, filename="best")
    # vuelve a calcular reward
    if mostrar_detalle_recompensa_promedio:
      print("   -Mejor Modelo: ", end="")
    newBest_avg_reward = compute_avg_return(env, dqnAg, cant_episodios_evaluacion, display=mostrar_detalle_recompensa_promedio)
    # compara
    if (last_avg_reward>newBest_avg_reward) or\
        ((last_avg_reward==newBest_avg_reward) and (avg_reward>best_avg_reward)):
        print("++ Se usa último modelo entrenado: Ultimo {:.3f} >= Mejor {:.3f}"
                .format(last_avg_reward, newBest_avg_reward))
        dqnAg.load_checkpoint(folder=tempCheckpointDir, filename="last")
    else:
        print("++ Se usa mejor modelo entrenado: Ultimo {:.3f} < Mejor {:.3f}"
                .format(last_avg_reward, newBest_avg_reward))
  print("")
  # vuelve a poner grabado de video a su estado anterior
  grabaVideoEnv = bak_grabaVideoEnv
else:
  print("No se ejecuta entrenamiento de Agente DQN.")

In [ ]:
#@title Grafica entrenamiento de Agente DQN
import matplotlib.pyplot as plt

plt.figure(figsize=(15,8))
plt.plot(rewards_per_episode)
plt.xlabel('Episodio')
plt.ylabel('Recompensa')
plt.title('Entrenamiento de Agente DQN')
plt.show()

In [ ]:
#@title Probar DQN Entrenado contra el Azar
cantidad_probar = 1 # @param {type:"integer"}
mostar_jugadas_agente = True # @param {type:"boolean"}

if dqnAg is not None:
  compararRtdosAgentes(cantidad_probar, envProblema,
                       dqnAg, randomAgent, descAg1="Agente DQN", descAg2="Azar",
                       mostrarRenderAg1=mostar_jugadas_agente, mostrarRenderAg2=False)


In [ ]:

#@title Cargar o Guardar los Agentes Q-Learning y DQN entrenados

# parámetros
directorio_modelos = '/content/gdrive/MyDrive/demosColab/demoRL/Modelos/QAtari' #@param {type:"string"}
accion_realizar = "-" #@param ["-", "Cargar Modelo", "Grabar Modelo"]

if accion_realizar != "-":

  # le agrega nombre del juego
  directorio_modelos = directorio_modelos + "_" + nombre_juego

  import joblib
  import os

  # Montar Drive
  from google.colab import drive
  drive.mount('/content/gdrive')


  if accion_realizar == "Grabar Modelo":

      # define configuración de agentes
      defArchivosConfig = []
      defArchivosConfig.append( [dqnAg, "/DQN_agent.joblib", "Agente DQN"] )

      # si no existe el directorio, lo crea
      if not os.path.isdir(directorio_modelos):
        os.makedirs(directorio_modelos)
      # guarda las clases auxiliares de los agentes entrenados
      for config in defArchivosConfig:
        ar = directorio_modelos+config[1]
        joblib.dump(config[0], ar)
        print("-" + str(config[2]) + " grabado en "+ str(ar))
        if "DQN" in config[2]:
          config[0].save_model(folder=directorio_modelos)

  elif accion_realizar == "Cargar Modelo":

      # define configuración de agentes
      defArchivosConfig = []
      defArchivosConfig.append( [None, "/DQN_agent.joblib", "Agente DQN"] )

      # carga la política del modelo --- el resto no se necesita cargar
      for i in range(len(defArchivosConfig)):
        config = defArchivosConfig[i]
        ar = directorio_modelos+config[1]
        if os.path.isfile(ar):
          infoRecup = joblib.load(ar)
          if i==0:
            dqnAg = infoRecup
          else:
              raise ValueError(str(i)+ " iteración no definida varialbe a recuperar!")
          print("-" + str(config[2]) + " recuperado de "+ str(ar))
        else:
          print("- No se encuentra el archivo ", ar)
